# Mapping Belief-Quantization Sweep (`J_M^*` vs `M`)

This notebook runs a mapping-focused sweep over `M` (belief quantization level), while keeping state quantization `n` fixed.

Goal: track how the optimal value `J_M^*` changes with increasing belief resolution.

Outputs are saved under `output/experiment2/belief_quantization_convergence_mapping_<timestamp>/`:
- incremental JSON checkpoints
- final CSV summary
- `J_M^*` vs `M` plot

In [1]:
import os
import sys
import json
import time
import subprocess
import ctypes
from datetime import datetime
from pathlib import Path

import numpy as onp
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('/global/home/hpc5656/SLAM')
sys.path.append(str(PROJECT_ROOT))
os.chdir(str(PROJECT_ROOT))
print('CWD:', Path.cwd())

%matplotlib inline

# Preload NVIDIA driver library first (matches working sweep notebooks).
try:
    ctypes.CDLL('/usr/lib64/libcuda.so.1', mode=ctypes.RTLD_GLOBAL)
    print('✓ Preloaded libcuda.so.1 from /usr/lib64')
except Exception as e:
    print(f'⚠ Could not preload libcuda.so.1: {e}')

# Ensure CUDA module env is loaded when CUDA_PATH is missing.
if 'CUDA_PATH' not in os.environ:
    try:
        result = subprocess.run(
            'module load cuda/12.2 && env',
            shell=True,
            executable='/bin/bash',
            capture_output=True,
            text=True,
            timeout=10,
        )
        if result.returncode == 0:
            for line in result.stdout.split('\n'):
                if '=' in line:
                    key, value = line.split('=', 1)
                    os.environ[key] = value
            print(f"✓ Modules loaded. CUDA_PATH: {os.environ.get('CUDA_PATH', 'Not set')}")
        else:
            raise RuntimeError('CUDA_PATH not set. Run: module load cuda/12.2')
    except Exception as e:
        raise RuntimeError(f'Failed to load CUDA module: {e}') from e

cuda_path = os.environ.get('CUDA_PATH')
if cuda_path:
    cuda_lib_paths = [
        os.path.join(cuda_path, 'lib64'),
        os.path.join(cuda_path, 'targets', 'x86_64-linux', 'lib'),
    ]
    current_ld = os.environ.get('LD_LIBRARY_PATH', '')
    ld_paths = current_ld.split(':') if current_ld else []

    for p in cuda_lib_paths:
        if os.path.exists(p) and p not in ld_paths:
            ld_paths.insert(0, p)

    if ld_paths:
        os.environ['LD_LIBRARY_PATH'] = ':'.join(ld_paths)
        print('✓ Updated LD_LIBRARY_PATH')

    for p in cuda_lib_paths:
        if not os.path.exists(p):
            continue
        for name in ['libcudart.so', 'libcudart.so.12', 'libnvrtc.so.12']:
            candidate = os.path.join(p, name)
            if os.path.exists(candidate):
                try:
                    ctypes.CDLL(candidate, mode=ctypes.RTLD_GLOBAL)
                    print(f'✓ Preloaded {name}')
                except Exception as e:
                    print(f'⚠ Could not preload {name}: {e}')
                break

print('\n=== Environment before backend import ===')
print(f"CUDA_PATH: {os.environ.get('CUDA_PATH', 'Not set')}")
print('=========================================\n')

from src.utils.array_backend import np, is_cupy
from src.belief_quantized.belief_mdp_n_M import BeliefMDP_n_M_Mapping
from src.classes.model import SingleIntegratorModel, RangeBearingSensor
from src.classes.mapping import LandmarkMap
from src.belief_quantized.value_iteration import ValueIteration


def to_numpy(x):
    return x.get() if hasattr(x, 'get') else onp.asarray(x)


if is_cupy:
    import cupy as cp
    print(f"Backend: CuPy (GPU) | CuPy {cp.__version__}, devices: {cp.cuda.runtime.getDeviceCount()}")
else:
    print('Backend: NumPy (CPU)')

CWD: /global/home/hpc5656/SLAM
✓ Preloaded libcuda.so.1 from /usr/lib64
✓ Modules loaded. CUDA_PATH: /cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v3/Core/cudacore/12.2.2
✓ Updated LD_LIBRARY_PATH
✓ Preloaded libcudart.so
✓ Preloaded libcudart.so

=== Environment before backend import ===
CUDA_PATH: /cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v3/Core/cudacore/12.2.2

✓ Using CuPy for GPU acceleration
✓ Using cupyx.scipy.spatial.KDTree
Backend: CuPy (GPU) | CuPy 14.0.0rc1, devices: 8


In [2]:
# ----------------------------
# Sweep configuration
# ----------------------------
n_fixed = 3
M_SWEEP = list(range(2, 8))   # M = 2..7
beta = 0.95
epsilon = 1e-6

obs_n = 3
action_n = 4
sigma_w = 0.1
sigma_r = 0.75
sigma_phi = 0.35

# Keep map discretization fixed.
map_n = 2
num_landmarks = 2

# Optional compute knobs
n_gpus = 8
j_batch_size = 3086
i_batch_size = 3086

# Reference state for J_M^* extraction (nearest quantized state index).
x_ref = onp.array([5.0, 5.0], dtype=float)

# Output/checkpoint paths
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
out_dir = PROJECT_ROOT / 'output' / 'experiment2' / f'belief_quantization_convergence_mapping_{stamp}'
out_dir.mkdir(parents=True, exist_ok=True)

checkpoint_path = out_dir / 'progress_belief_quantization_sweep_mapping.json'
print('Output dir:', out_dir)
print('Fixed n:', n_fixed)
print('Sweep M:', M_SWEEP)


def build_mdp(M: int) -> BeliefMDP_n_M_Mapping:
    landmark_map = LandmarkMap(
        x_min=0.0,
        x_max=10.0,
        y_min=0.0,
        y_max=10.0,
        n=map_n,
        num_landmarks=num_landmarks,
    )
    motion_model = SingleIntegratorModel(i_x=5.0, i_y=5.0, dt=1.0, max_v=4.0)
    sensor = RangeBearingSensor(r_max=4.0, epsilon=0.1, sigma_r=sigma_r, sigma_phi=sigma_phi)
    cov_y = onp.diag(onp.tile([sigma_r**2, sigma_phi**2], landmark_map.num_candidate_landmarks))

    return BeliefMDP_n_M_Mapping(
        M=M,
        β=beta,
        n=n_fixed,
        motion_model=motion_model,
        measurement_model=sensor,
        obstacles=[],
        _map=landmark_map,
        sigma_w=sigma_w,
        cov_y=cov_y,
        obs_n=obs_n,
        action_n=action_n,
        j_batch_size=j_batch_size,
        i_batch_size=i_batch_size,
        n_gpus=n_gpus,
    )


def reference_indices(mdp: BeliefMDP_n_M_Mapping):
    # Reference belief: projection of uniform prior over all map hypotheses.
    all_maps = to_numpy(mdp.all_maps_3d)
    codebook = to_numpy(mdp.BQ.Π_n_M)
    prior = onp.full(all_maps.shape[0], 1.0 / all_maps.shape[0], dtype=float)
    b_idx = int(onp.argmin(onp.linalg.norm(codebook - prior[None, :], axis=1)))

    # Reference state: nearest quantized state to x_ref.
    X_n = to_numpy(mdp.SQ.X_n)
    x_idx = int(onp.argmin(onp.linalg.norm(X_n - x_ref[None, :], axis=1)))
    return b_idx, x_idx

Output dir: /global/home/hpc5656/SLAM/output/experiment2/belief_quantization_convergence_mapping_20260310_123827
Fixed n: 3
Sweep M: [2, 3, 4, 5, 6, 7]


In [ ]:
# ----------------------------
# Run sweep
# ----------------------------
if checkpoint_path.exists():
    with open(checkpoint_path, 'r') as f:
        records = json.load(f)
    print(f'Loaded {len(records)} checkpointed result(s).')
else:
    records = []

completed_M = {int(r['M']) for r in records if 'error' not in r}

for M in M_SWEEP:
    if M in completed_M:
        print(f'M={M}: already done, skipping.')
        continue

    print(f'\n=== M={M} ===')
    t0 = time.time()

    rec = {'M': int(M), 'n_fixed': int(n_fixed)}
    try:
        mdp = build_mdp(M)
        rec['m_n'] = int(mdp.SQ.m_n)
        rec['cardinality'] = int(mdp.BQ.cardinality)

        vi = ValueIteration(mdp, epsilon=epsilon)
        loaded = bool(vi.load_results())
        if not loaded:
            vi.run(verbose=False)
            vi.save_results()

        V = to_numpy(vi.V)
        b_idx, x_idx = reference_indices(mdp)

        rec.update({
            'used_cached_vi': loaded,
            'iterations': int(vi.iteration_count),
            'b_ref_index': int(b_idx),
            'x_ref_index': int(x_idx),
            'J_star_ref': float(V[b_idx, x_idx]),
            'V_mean': float(V.mean()),
            'V_min': float(V.min()),
            'V_max': float(V.max()),
        })

        print(
            f"done M={M} | n={n_fixed} | m_n={rec['m_n']} | |Pi|={rec['cardinality']} | "
            f"iters={rec['iterations']} | J*={rec['J_star_ref']:.6f}"
        )

    except Exception as e:
        rec['error'] = str(e)
        print(f'M={M} failed: {e}')

    rec['elapsed_s'] = float(time.time() - t0)
    records = [r for r in records if int(r.get('M', -1)) != M] + [rec]

    with open(checkpoint_path, 'w') as f:
        json.dump(sorted(records, key=lambda r: int(r.get('M', -1))), f, indent=2)

print('\nSweep finished (or checkpoint updated).')
checkpoint_path


=== M=2 ===
Loaded cached T_mat from /global/home/hpc5656/SLAM/cache/T_mat/T_mat_n3_map2x2_max4.0_736f076e.npz
  Checking cache file: Q_n_n3_nmap2_obs3_map2x2_L2_cb49f02b.npz
  ✓ Cache metadata matches, loading Q_n
Loaded cached Q_n from /global/home/hpc5656/SLAM/cache/Q_n/Q_n_n3_nmap2_obs3_map2x2_L2_cb49f02b.npz
  ✓ Cache validation passed
  ✓ Loaded codebook from cache: cache/belief_quantizer/belief_quantizer_M2_N16.npz
Loaded cached p_n_M from /global/home/hpc5656/SLAM/cache/MAP/p_n_M_mapping/p_n_M_mapping_M2_n3_nmap2_map2x2_max4.0_86206a7a.npz
Loaded cached c_n_M from /global/home/hpc5656/SLAM/cache/MAP/cost_mapping/cost_mapping_M2_n3_nmap2_map2x2_max4.0_7e48733c.npz
Saved value iteration results to /global/home/hpc5656/SLAM/cache/MAP/value_iteration/value_iteration_mapping_M2_n3_beta0.95_eps1e-06_map2x2_max4.0_b3a7e178.npz
  File size: 0.00 MB
  Iterations: 277
  Final max |V - V_old|: 9.90e-07
done M=2 | n=3 | m_n=9 | |Pi|=136 | iters=277 | J*=28.538407

=== M=3 ===
Loaded cache

/global/home/hpc5656/SLAM/src/classes/belief_mdp_n_M.py:1884: RuntimeWarning: divide by zero encountered in log
  log_π_all_cpu = _numpy.log(_numpy.maximum(Π_cpu, 1e-300))


Computing p_n_M for mapping (quantized η_n_batch, sparse)...
  States: 1,534,896 = m_n=9 × cardinality=170544, n_u=16
  Obs pruning: 100 total obs, valid per x_next: min=81, max=100, mean=98
  Multi-GPU: 8 devices, 16 actions → 2 rounds
  Round 1/2: actions [1, 2, 3, 4, 5, 6, 7, 8] on GPUs 0..7


Action 1/16 [GPU 0]:   0%|          | 0/1534896 [00:00<?, ?it/s]

Action 3/16 [GPU 2]:   0%|          | 0/1534896 [00:00<?, ?it/s]

...

Action 7/16 [GPU 6]:   0%|          | 0/1534896 [00:00<?, ?it/s]

Action 2/16 [GPU 1]:   0%|          | 0/1534896 [00:00<?, ?it/s]

Action 5/16 [GPU 4]:   0%|          | 0/1534896 [00:00<?, ?it/s]

Action 6/16 [GPU 5]:   0%|          | 0/1534896 [00:00<?, ?it/s]

Action 8/16 [GPU 7]:   0%|          | 0/1534896 [00:00<?, ?it/s]

In [ ]:
# ----------------------------
# Summarize and plot
# ----------------------------
df = pd.DataFrame(sorted(records, key=lambda r: int(r.get('M', -1))))

ok_df = df[df.get('error').isna()] if 'error' in df.columns else df
if ok_df.empty:
    raise RuntimeError('No successful runs found. Check checkpoint JSON for errors.')

ok_df = ok_df.sort_values('M').reset_index(drop=True)
ok_df['delta_J_star_ref'] = ok_df['J_star_ref'].diff()

summary_csv = out_dir / (
    f"summary_belief_quantization_mapping_n{n_fixed}_M{min(M_SWEEP)}-{max(M_SWEEP)}"
    f"_obs{obs_n}_act{action_n}_beta{beta}_sr{sigma_r}_sp{sigma_phi}.csv"
)
ok_df.to_csv(summary_csv, index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

axes[0].plot(ok_df['M'], ok_df['J_star_ref'], marker='o', linewidth=2)
axes[0].set_xlabel('Belief quantization level M')
axes[0].set_ylabel(r'$J_M^*$ at reference (b,x)')
axes[0].set_title('Value convergence trace')
axes[0].grid(alpha=0.3)

axes[1].plot(ok_df['M'], ok_df['delta_J_star_ref'].fillna(0.0), marker='o', linewidth=2)
axes[1].axhline(0.0, color='k', linestyle='--', linewidth=1)
axes[1].set_xlabel('Belief quantization level M')
axes[1].set_ylabel(r'$J_M^* - J_{M-1}^*$')
axes[1].set_title('Incremental change by belief quantization level')
axes[1].grid(alpha=0.3)

fig.suptitle(
    f"Mapping belief-quantization sweep | n={n_fixed}, obs_n={obs_n}, action_n={action_n}, "
    f"beta={beta}, sr={sigma_r}, sp={sigma_phi}"
)
fig.tight_layout()

plot_png = out_dir / (
    f"Jstar_vs_M_mapping_n{n_fixed}_M{min(M_SWEEP)}-{max(M_SWEEP)}"
    f"_obs{obs_n}_act{action_n}_beta{beta}_sr{sigma_r}_sp{sigma_phi}.png"
)
fig.savefig(plot_png, bbox_inches='tight', dpi=180)
plt.show()

print('Saved summary:', summary_csv)
print('Saved plot:', plot_png)
ok_df